<a href="https://colab.research.google.com/github/albertoaguirreca-eng/arqia-triage-core/blob/main/notebooks/jtm_simulacion_piloto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
from google.colab import files

# 1. Asegurar carpeta data/ (estándar de tu repo arqia-triage-core)
os.makedirs("data", exist_ok=True)
np.random.seed(42)

departamentos = {
    'Operações Portuárias': 15,
    'Gestão de Frota': 15,
    'Direção e Comercial': 12
}

semanas = [f'W0{i}' for i in range(1, 9)]
retos_pt = [
    "Fechar planeamento semanal sem atrasos",
    "Delegar tarefas operacionais críticas",
    "Manter foco nas prioridades do trimestre",
    "Melhorar coordenação entre turnos",
    "Reduzir reuniões improdutivas"
]

registros = []
user_idx = 1

for dept, n_users in departamentos.items():
    for u in range(n_users):
        user_hash = f"JTM_{user_idx:03d}"
        user_idx += 1

        perfil = "estable"
        if dept == 'Operações Portuárias' and u < 6:
            perfil = "desgaste_progresivo"  # Caída en W06-W08
        elif dept == 'Gestão de Frota' and u == 0:
            perfil = "alerta_critica_3sem"  # <= 3 por 3 semanas (activa protocolo privado)
        elif dept == 'Direção e Comercial' and u < 3:
            perfil = "varianza_cero"        # Marca siempre 8 (sesgo automático)

        for w_idx, sem in enumerate(semanas):
            if perfil == "varianza_cero":
                pessoal = saude = energia = profissional = desafio = 8
            elif perfil == "alerta_critica_3sem" and w_idx >= 5:
                pessoal = np.random.randint(3, 5)
                saude = np.random.randint(2, 4)
                energia = np.random.randint(2, 4)
                profissional = np.random.randint(2, 4)
                desafio = np.random.randint(3, 5)
            elif perfil == "desgaste_progresivo" and w_idx >= 5:
                pessoal = np.random.randint(5, 7)
                saude = np.random.randint(4, 6)
                energia = np.random.randint(3, 5)
                profissional = np.random.randint(3, 5)
                desafio = np.random.randint(4, 6)
            else:
                pessoal = int(np.clip(np.random.normal(7.2, 0.8), 1, 9))
                saude = int(np.clip(np.random.normal(7.1, 0.8), 1, 9))
                energia = int(np.clip(np.random.normal(7.0, 0.9), 1, 9))
                profissional = int(np.clip(np.random.normal(7.1, 0.8), 1, 9))
                desafio = int(np.clip(np.random.normal(7.0, 0.9), 1, 9))

            # Cálculo preliminar de ISP (escala 0 a 100)
            isp_score = round(((pessoal*0.15 + saude*0.25 + energia*0.25 + profissional*0.25 + desafio*0.10) / 9) * 100, 1)

            registros.append({
                'user_hash': user_hash,
                'departamento': dept,
                'semana': sem,
                'pessoal': pessoal,
                'saude': saude,
                'energia': energia,
                'profissional': profissional,
                'desafio': desafio,
                'isp_score': isp_score,
                'desafio_texto': np.random.choice(retos_pt)
            })

df_jtm = pd.DataFrame(registros)
ruta_csv = 'data/jtm_checkin_sintetico_v3.csv'
df_jtm.to_csv(ruta_csv, index=False, encoding='utf-8-sig')

print(f"✅ Dataset generado: {len(df_jtm)} registros ({user_idx - 1} colaboradores x 8 semanas).")
print("\nPromedio ISP por departamento en la última semana (W08):")
print(df_jtm[df_jtm['semana'] == 'W08'].groupby('departamento')['isp_score'].mean().round(1))

# Descargar automáticamente a tu computador para enviarlo a Camilo (Pymetech)
files.download(ruta_csv)

✅ Dataset generado: 336 registros (42 colaboradores x 8 semanas).

Promedio ISP por departamento en la última semana (W08):
departamento
Direção e Comercial     77.7
Gestão de Frota         72.8
Operações Portuárias    63.3
Name: isp_score, dtype: float64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>